In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║               BioEnhance-Agent — Complete Pipeline           ║
# ║   Install → Load → EDA → Labels (no leakage) → Benchmark →   ║
# ║           Optuna → Evaluate → SHAP → Save → Download         ║
# ╚══════════════════════════════════════════════════════════════╝

# ── 1. Install (clean — no force-reinstall, avoids scipy conflicts) ──
!pip install -q xgboost lightgbm catboost optuna shap scikit-learn

# ── 2. Imports ────────────────────────────────────────────────
import glob, io, os, re, warnings, time, zipfile
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, f1_score, roc_auc_score
)
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
import xgboost as xgb
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
import shap
import joblib

plt.rcParams['figure.dpi'] = 120
sns.set_style('whitegrid')
PALETTE = ['#2563EB', '#16A34A', '#DC2626', '#D97706', '#7C3AED']
print('✅ Packages loaded.')

# ── 3. Upload & Load Dataset ─────────────────────────────────
from google.colab import files as colab_files
print('\nUpload curated-solubility-dataset.csv or .zip:')
uploaded = colab_files.upload()
filename = list(uploaded.keys())[0]

if filename.endswith('.zip'):
    os.makedirs('dataset', exist_ok=True)
    with zipfile.ZipFile(io.BytesIO(uploaded[filename])) as z:
        z.extractall('dataset')
    csv_path = glob.glob('dataset/*.csv')[0]
    df = pd.read_csv(csv_path)
else:
    df = pd.read_csv(io.BytesIO(uploaded[filename]))

print(f'✅ Loaded — {df.shape[0]:,} compounds · {df.shape[1]} columns')

# ── 4. EDA ────────────────────────────────────────────────────
print(f'\nMissing values : {df.isnull().sum().sum()}')
print(f'Duplicates     : {df.duplicated().sum()}')

fig, axes = plt.subplots(2, 3, figsize=(15, 8))

axes[0,0].hist(df['Solubility'], bins=60, color='#2563EB', alpha=0.85, edgecolor='white')
for val, col, lbl in [(-2,'#DC2626','−2'),(-4,'#D97706','−4'),(-6,'#7C3AED','−6')]:
    axes[0,0].axvline(val, color=col, lw=1.8, ls='--', label=f'LogS={lbl}')
axes[0,0].set_xlabel('LogS'); axes[0,0].set_title('Aqueous Solubility', fontweight='bold')
axes[0,0].legend(fontsize=8)

axes[0,1].hist(df['MolLogP'], bins=60, color='#16A34A', alpha=0.85, edgecolor='white')
axes[0,1].set_xlabel('MolLogP'); axes[0,1].set_title('Lipophilicity', fontweight='bold')

axes[0,2].hist(df['MolWt'], bins=60, color='#D97706', alpha=0.85, edgecolor='white')
axes[0,2].set_xlabel('MW (Da)'); axes[0,2].set_title('Molecular Weight', fontweight='bold')

axes[1,0].hist(df['TPSA'], bins=60, color='#7C3AED', alpha=0.85, edgecolor='white')
axes[1,0].set_xlabel('TPSA (Å²)'); axes[1,0].set_title('TPSA', fontweight='bold')

axes[1,1].hist(df['NumHDonors'], bins=20, color='#DC2626', alpha=0.85, edgecolor='white')
axes[1,1].set_xlabel('H-Bond Donors'); axes[1,1].set_title('H-Bond Donors', fontweight='bold')

axes[1,2].scatter(df['MolLogP'], df['Solubility'], alpha=0.12, s=6, color='#16A34A')
for val, col in [(-2,'#DC2626'),(-4,'#D97706'),(-6,'#7C3AED')]:
    axes[1,2].axhline(val, color=col, lw=1.5, ls='--')
axes[1,2].set_xlabel('MolLogP'); axes[1,2].set_ylabel('LogS')
axes[1,2].set_title('LogS vs MolLogP', fontweight='bold')

plt.suptitle('AqSolDB — EDA Overview', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_eda.png', bbox_inches='tight')
plt.show()
print('✅ fig_eda.png')

# ── 5. Label Engineering (NO LEAKAGE) ────────────────────────
# Solubility used ONLY to derive labels — excluded from features
def assign_strategy(row):
    s, p = row['Solubility'], row['MolLogP']
    if   s > -2:             return 'No_Enhancement_Needed'
    elif s > -4 and p < 3:  return 'Amorphous_Solid_Dispersion'
    elif s > -4 and p >= 3: return 'Lipid_Based_Formulation'
    elif s > -6:             return 'Nanosuspension'
    else:                    return 'Cocrystal_or_Cyclodextrin'

df['Strategy'] = df.apply(assign_strategy, axis=1)
counts = df['Strategy'].value_counts()

print('\n=== Strategy Distribution ===')
for s, n in counts.items():
    print(f'  {s:<35} {n:>5,}  ({n/len(df)*100:.1f}%)')

fig, ax = plt.subplots(figsize=(10, 3.5))
bars = ax.barh(counts.index, counts.values, color=PALETTE, height=0.55, edgecolor='white')
for bar, v in zip(bars, counts.values):
    ax.text(bar.get_width()+30, bar.get_y()+bar.get_height()/2,
            f'{v:,} ({v/len(df)*100:.0f}%)', va='center', fontsize=10)
ax.set_xlim(0, counts.max()*1.3)
ax.set_xlabel('Number of Compounds')
ax.set_title('Bioenhancement Strategy Distribution', fontweight='bold')
plt.tight_layout()
plt.savefig('fig_class_distribution.png', bbox_inches='tight')
plt.show()
print('✅ fig_class_distribution.png')

# ── 6. Feature Matrix (12 descriptors — NO Solubility) ───────
FEATURES = [
    'MolLogP', 'MolWt', 'NumHDonors', 'NumHAcceptors',
    'NumRotatableBonds', 'NumAromaticRings', 'RingCount',
    'TPSA', 'HeavyAtomCount', 'MolMR', 'LabuteASA', 'BertzCT',
]

missing_cols = [c for c in FEATURES if c not in df.columns]
if missing_cols:
    print(f'⚠️  Missing columns: {missing_cols}')
else:
    print(f'✅ All {len(FEATURES)} features confirmed (no Solubility — leakage-free).')

fig, ax = plt.subplots(figsize=(11, 9))
mask = np.triu(np.ones_like(df[FEATURES].corr(), dtype=bool))
sns.heatmap(df[FEATURES].corr(), mask=mask, annot=True, fmt='.2f',
            cmap='RdBu_r', center=0, ax=ax, linewidths=0.4, annot_kws={'size': 7})
ax.set_title('Feature Correlation Matrix (12 descriptors, no leakage)', fontweight='bold')
plt.tight_layout()
plt.savefig('fig_correlation.png', bbox_inches='tight')
plt.show()
print('✅ fig_correlation.png')

df_clean = df[FEATURES + ['Strategy']].dropna().copy()
X  = df_clean[FEATURES].astype(float)
le = LabelEncoder()
y  = le.fit_transform(df_clean['Strategy'])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print(f'\n✅ X shape  : {X.shape}')
print(f'✅ Train    : {len(X_train):,} | Test: {len(X_test):,}')
print(f'✅ No leakage — Solubility excluded from features')
print(f'Classes     : {list(le.classes_)}')

# ── 7. Model Benchmark ───────────────────────────────────────
print('\n' + '='*54)
print('  Model Benchmark — 5-Fold CV (default hyperparameters)')
print('='*54)

MODELS = {
    'Logistic Regression': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', LogisticRegression(max_iter=1000, random_state=42, n_jobs=-1))
    ]),
    'Random Forest': RandomForestClassifier(
        n_estimators=200, random_state=42, n_jobs=-1
    ),
    'XGBoost': XGBClassifier(
        n_estimators=200, max_depth=5, learning_rate=0.1,
        eval_metric='mlogloss', random_state=42, n_jobs=-1
    ),
    'LightGBM': LGBMClassifier(
        n_estimators=200, num_leaves=31, n_jobs=1,
        verbose=-1, random_state=42
    ),
    'CatBoost': CatBoostClassifier(
        iterations=200, random_state=42, verbose=0
    ),
}

benchmark_results = []
for name, m in MODELS.items():
    print(f'  {name} ...', end=' ', flush=True)
    t0 = time.time()
    scores = cross_val_score(m, X_train, y_train,
                             cv=CV, scoring='f1_weighted', n_jobs=-1)
    elapsed = time.time() - t0
    benchmark_results.append({
        'Model': name, 'CV F1 Mean': round(scores.mean(), 4),
        'CV F1 Std': round(scores.std(), 4), 'Time (s)': round(elapsed, 1),
    })
    print(f'F1 = {scores.mean():.4f} ± {scores.std():.4f}  ({elapsed:.1f}s)')

bench_df = pd.DataFrame(benchmark_results).sort_values('CV F1 Mean', ascending=False)
display(bench_df)

fig, ax = plt.subplots(figsize=(9, 4))
colors = ['#2563EB' if i == 0 else '#94A3B8' for i in range(len(bench_df))]
bars = ax.barh(bench_df['Model'], bench_df['CV F1 Mean'],
               color=colors, height=0.5, edgecolor='white')
ax.set_xlim(0, 1.05)
for bar, v in zip(bars, bench_df['CV F1 Mean']):
    ax.text(bar.get_width()+0.01, bar.get_y()+bar.get_height()/2,
            f'{v:.4f}', va='center', fontsize=9, fontweight='bold')
ax.set_xlabel('CV F1-weighted (5-fold)')
ax.set_title('Model Benchmark — Default Hyperparameters (no leakage)', fontweight='bold')
plt.tight_layout()
plt.savefig('fig_benchmark.png', bbox_inches='tight')
plt.show()
print('✅ fig_benchmark.png')
print(f'\n🏆 Best baseline: {bench_df.iloc[0]["Model"]}')
print('→  Proceeding with XGBoost + Optuna (SHAP-compatible, literature standard)')

# ── 8. Optuna Hyperparameter Optimisation ────────────────────
print('\n' + '='*54)
print('  Optuna — Bayesian Search (50 trials)')
print('='*54)

def objective(trial):
    params = dict(
        n_estimators     = trial.suggest_int  ('n_estimators',     100, 600),
        max_depth        = trial.suggest_int  ('max_depth',          3,   8),
        learning_rate    = trial.suggest_float('learning_rate',  0.01, 0.3, log=True),
        subsample        = trial.suggest_float('subsample',       0.6, 1.0),
        colsample_bytree = trial.suggest_float('colsample_bytree', 0.6, 1.0),
        min_child_weight = trial.suggest_int  ('min_child_weight',   1,  10),
        gamma            = trial.suggest_float('gamma',            0.0, 0.5),
        reg_alpha        = trial.suggest_float('reg_alpha',    1e-8, 1.0, log=True),
        reg_lambda       = trial.suggest_float('reg_lambda',   1e-8, 1.0, log=True),
        eval_metric      = 'mlogloss', random_state = 42, n_jobs = -1,
    )
    return cross_val_score(XGBClassifier(**params), X_train, y_train,
                           cv=CV, scoring='f1_weighted', n_jobs=-1).mean()

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=50, show_progress_bar=True)

print(f'\n✅ Best CV F1 : {study.best_value:.4f}')
for k, v in study.best_params.items():
    print(f'  {k:<22}: {v}')

best_params = study.best_params.copy()
best_params.update(dict(eval_metric='mlogloss', random_state=42, n_jobs=-1))
tuned_model = XGBClassifier(**best_params)
tuned_model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)
print('\n✅ Tuned model trained.')

trial_vals = [t.value for t in study.trials]
best_curve  = pd.Series(trial_vals).cummax()

fig, ax = plt.subplots(figsize=(10, 3.5))
ax.plot(trial_vals, 'o', color='#2563EB', alpha=0.35, ms=4, label='Trial F1')
ax.plot(best_curve.values, color='#DC2626', lw=2, label='Best so far')
ax.set_xlabel('Trial'); ax.set_ylabel('CV F1-weighted')
ax.set_title('Optuna Optimisation History', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig('fig_optuna.png', bbox_inches='tight')
plt.show()
print('✅ fig_optuna.png')

# ── 9. Evaluation ─────────────────────────────────────────────
print('\n' + '='*54)
print('  Model Evaluation — Holdout Test Set (20%)')
print('='*54)

y_pred = tuned_model.predict(X_test)
y_prob = tuned_model.predict_proba(X_test)
cm     = confusion_matrix(y_test, y_pred)
cm_pct = cm.astype(float) / cm.sum(axis=1, keepdims=True) * 100

acc = accuracy_score(y_test, y_pred)
f1  = f1_score(y_test, y_pred, average='weighted')
try:
    auc = roc_auc_score(y_test, y_prob, multi_class='ovr', average='weighted')
except:
    auc = None

print(f'\n  Test Accuracy          : {acc:.4f}  ({acc*100:.2f}%)')
print(f'  Weighted F1-Score      : {f1:.4f}')
if auc: print(f'  Weighted ROC-AUC (OvR) : {auc:.4f}')
print(f'  Optuna Best CV F1      : {study.best_value:.4f}')
print()
print(classification_report(y_test, y_pred, target_names=le.classes_, digits=3))

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
short = [c[:12] for c in le.classes_]
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=short,
            yticklabels=le.classes_, ax=axes[0], linewidths=0.5)
axes[0].set_title('Confusion Matrix (Counts)', fontweight='bold')
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('True')
axes[0].tick_params(axis='x', rotation=30)
sns.heatmap(cm_pct, annot=True, fmt='.1f', cmap='Blues', xticklabels=short,
            yticklabels=le.classes_, ax=axes[1], linewidths=0.5, vmin=0, vmax=100)
axes[1].set_title('Confusion Matrix — Row-Normalised Recall (%)', fontweight='bold')
axes[1].set_xlabel('Predicted'); axes[1].set_ylabel('True')
axes[1].tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.savefig('fig_confusion.png', bbox_inches='tight')
plt.show()
print('✅ fig_confusion.png')

# ── 10. SHAP (native XGBoost — avoids version conflicts) ─────
print('\nComputing SHAP values...')

X_shap    = X_test.astype(float).reset_index(drop=True).iloc[:300]
n_classes = len(le.classes_)
dmat      = xgb.DMatrix(X_shap, feature_names=FEATURES)
raw       = tuned_model.get_booster().predict(dmat, pred_contribs=True)

if raw.ndim == 3:
    s0, s1, s2 = raw.shape
    if s1 == n_classes:
        shap_3d = raw[:, :, :-1]
        shap_list = [shap_3d[:, i, :] for i in range(n_classes)]
    else:
        shap_3d = raw[:, :-1, :]
        shap_list = [shap_3d[:, :, i] for i in range(n_classes)]
elif raw.ndim == 2:
    n_samples = len(X_shap)
    shap_2d   = raw[:, :-1].reshape(n_samples, n_classes, -1)
    shap_list = [shap_2d[:, i, :] for i in range(n_classes)]
else:
    raise ValueError(f'Unexpected SHAP shape: {raw.shape}')

assert shap_list[0].shape == (len(X_shap), len(FEATURES))
print('✅ SHAP shapes verified.')

mean_abs = np.mean([np.abs(s).mean(axis=0) for s in shap_list], axis=0)
feat_imp = pd.Series(mean_abs, index=FEATURES).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(9, 5))
feat_imp.plot(kind='barh', ax=ax, color='#2563EB', alpha=0.85)
ax.set_xlabel('Mean |SHAP value|')
ax.set_title('SHAP — Global Feature Importance', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_shap_global.png', bbox_inches='tight')
plt.show()
print('✅ fig_shap_global.png')

asd_idx  = list(le.classes_).index('Amorphous_Solid_Dispersion')
asd_shap = shap_list[asd_idx]
shap.summary_plot(asd_shap, X_shap, feature_names=FEATURES,
                  max_display=len(FEATURES), show=False)
plt.title('SHAP Beeswarm — Amorphous Solid Dispersion', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_shap_beeswarm_ASD.png', bbox_inches='tight')
plt.show()
print('✅ fig_shap_beeswarm_ASD.png')

ibuprofen = pd.DataFrame([{
    'MolLogP': 3.97, 'MolWt': 206.3, 'NumHDonors': 1,
    'NumHAcceptors': 1, 'NumRotatableBonds': 4, 'NumAromaticRings': 1,
    'RingCount': 1, 'TPSA': 37.3, 'HeavyAtomCount': 15,
    'MolMR': 53.5, 'LabuteASA': 82.1, 'BertzCT': 196.4,
}]).astype(float)

pred_cls  = tuned_model.predict(ibuprofen)[0]
pred_prob = tuned_model.predict_proba(ibuprofen)[0]
pred_name = le.inverse_transform([pred_cls])[0]

print(f'\n=== Ibuprofen (BCS Class II reference) ===')
print(f'Predicted : {pred_name}  ({pred_prob[pred_cls]*100:.1f}% confidence)')
for cls, p in sorted(zip(le.classes_, pred_prob), key=lambda x: -x[1]):
    print(f'  {cls:<35} {"█"*int(p*28)} {p:.3f}')

dmat_ibup = xgb.DMatrix(ibuprofen, feature_names=FEATURES)
raw_ibup  = tuned_model.get_booster().predict(dmat_ibup, pred_contribs=True)
if raw_ibup.ndim == 3:
    s0, s1, s2 = raw_ibup.shape
    ibup_shap_vals = raw_ibup[0, pred_cls, :-1] if s1 == n_classes else raw_ibup[0, :-1, pred_cls]
else:
    ibup_shap_vals = raw_ibup[:, :-1].reshape(1, n_classes, -1)[0, pred_cls, :]

feat_shap = pd.Series(ibup_shap_vals, index=FEATURES).sort_values()
colors    = ['#DC2626' if v > 0 else '#2563EB' for v in feat_shap]

fig, ax = plt.subplots(figsize=(8, 5))
feat_shap.plot(kind='barh', ax=ax, color=colors, alpha=0.85)
ax.axvline(0, color='black', lw=0.8)
ax.set_xlabel('SHAP value')
ax.set_title(f'SHAP — Ibuprofen → {pred_name}', fontweight='bold')
plt.tight_layout()
plt.savefig('fig_shap_waterfall_ibuprofen.png', bbox_inches='tight')
plt.show()
print('✅ fig_shap_waterfall_ibuprofen.png')

# ── 11. Summary Dashboard ─────────────────────────────────────
fig = plt.figure(figsize=(17, 10))
fig.suptitle('BioEnhance-Agent — Results Dashboard\nAqSolDB · XGBoost + Optuna · SHAP · No Leakage',
             fontsize=13, fontweight='bold')
gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.38)

ax_a = fig.add_subplot(gs[0, 0])
metrics = {'Accuracy': acc, 'F1 (weighted)': f1, 'Optuna CV F1': study.best_value}
if auc: metrics['ROC-AUC (OvR)'] = auc
bars_a = ax_a.barh(list(metrics.keys()), list(metrics.values()),
                   color=PALETTE[:len(metrics)], height=0.45)
ax_a.set_xlim(0, 1.12)
ax_a.axvline(0.9, color='gray', ls='--', alpha=0.5)
for bar, v in zip(bars_a, metrics.values()):
    ax_a.text(bar.get_width()+0.01, bar.get_y()+bar.get_height()/2,
              f'{v:.3f}', va='center', fontsize=10, fontweight='bold')
ax_a.set_title('Model Performance', fontweight='bold')

ax_b = fig.add_subplot(gs[0, 1])
c = df_clean['Strategy'].value_counts()
ax_b.pie(c.values, labels=[x[:14] for x in c.index], autopct='%1.0f%%',
         colors=PALETTE, startangle=90, textprops={'fontsize': 7.5})
ax_b.set_title('Class Balance', fontweight='bold')

ax_c = fig.add_subplot(gs[0, 2])
feat_imp.sort_values().plot(kind='barh', ax=ax_c, color='#2563EB', alpha=0.85)
ax_c.set_title('SHAP Feature Importance', fontweight='bold')
ax_c.set_xlabel('Mean |SHAP|')

ax_d = fig.add_subplot(gs[1, :2])
sns.heatmap(cm_pct, annot=True, fmt='.1f', cmap='Blues',
            xticklabels=[c[:13] for c in le.classes_],
            yticklabels=le.classes_, ax=ax_d, linewidths=0.4)
ax_d.set_title('Confusion Matrix — Row-Normalised Recall (%)', fontweight='bold')
ax_d.set_xlabel('Predicted'); ax_d.tick_params(axis='x', rotation=30)

ax_e = fig.add_subplot(gs[1, 2])
ax_e.plot(trial_vals, 'o', color='#2563EB', alpha=0.3, ms=3, label='Trial F1')
ax_e.plot(best_curve.values, color='#DC2626', lw=2, label='Best so far')
ax_e.set_xlabel('Trial'); ax_e.set_ylabel('CV F1-weighted')
ax_e.set_title('Optuna History', fontweight='bold'); ax_e.legend(fontsize=8)

plt.savefig('fig_summary_dashboard.png', bbox_inches='tight', dpi=150)
plt.show()
print('✅ fig_summary_dashboard.png')

# ── 12. Save Model Artifacts ──────────────────────────────────
joblib.dump(tuned_model, 'bioenhance_xgb_model.pkl')
joblib.dump(le,          'bioenhance_label_encoder.pkl')
joblib.dump(FEATURES,    'bioenhance_features.pkl')
print('\n✅ Model artifacts saved.')

# ── 13. Download All Outputs ──────────────────────────────────
print('\nDownloading all output files ...')
output_files = [
    'bioenhance_xgb_model.pkl', 'bioenhance_label_encoder.pkl', 'bioenhance_features.pkl',
    'fig_eda.png', 'fig_correlation.png', 'fig_class_distribution.png',
    'fig_benchmark.png', 'fig_optuna.png', 'fig_confusion.png',
    'fig_shap_global.png', 'fig_shap_beeswarm_ASD.png',
    'fig_shap_waterfall_ibuprofen.png', 'fig_summary_dashboard.png',
]
for f in output_files:
    if os.path.exists(f):
        colab_files.download(f)
        print(f'  ✅ {f}')
    else:
        print(f'  ⚠️  {f} not found')

print('\n' + '='*54)
print('  🎉 BioEnhance-Agent — Complete')
print(f'     Accuracy : {acc*100:.2f}%  |  F1: {f1:.4f}  |  AUC: {auc:.4f}')
print(f'     Figures  : 10 PNG files  |  Models: 3 PKL files')
print('='*54)